In [ ]:
!pip install torch-geometric
!pip install rank_bm25
!pip install rouge

In [2]:
import os
import json
from pprint import pprint
import torch
import torch.nn as nn
import torch.nn.functional as F
import networkx as nx
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, GraphSAGE, GATConv, GATv2Conv
from sentence_transformers import SentenceTransformer, util
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
from tqdm import tqdm
from collections import Counter
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from rouge import Rouge
from transformers import T5ForConditionalGeneration, T5Tokenizer

In [6]:
folder_path = "/data"

In [8]:
# file_path = os.path.join(folder_path, "data_ids_april7")
with open(folder_path + "/dev.json" , "r", encoding = "utf-8") as file:
  eval_data = json.load(file)
with open(folder_path + "/train.json" , "r", encoding = "utf-8") as file:
  train_data = json.load(file)

In [ ]:
print(len(train_data), len(eval_data))

In [ ]:
print(train_data[0].keys())
# entity_ids': '', 'supporting_facts': [], 'evidences': [], 'answer': '', 'evidences_id': [], 'answer_id': ''}
print(eval_data[0].keys())

# GOP Construction

In [9]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
class WikiHopDataset:
    def __init__(self, data_path):
        self.data = data_path
        self.sbert = SentenceTransformer('all-mpnet-base-v2')

    def create_passages(self, context):
        return {title: ' '.join(sents) for title, sents in context}

    def build_graph(self, context):
        passages = self.create_passages(context)
        titles = list(passages.keys())

        # Feature of a node is embedding of the passage that the node represents (each node is a passage [GOP])
        embeds = self.sbert.encode(list(passages.values()))
        node_features = torch.tensor(embeds, dtype = torch.float)

        edge_indices = []

        # 1. Semantic edges
        sim_matrix = np.inner(embeds, embeds)
        for i in range(len(titles)):
            for j in range(i + 1, len(titles)):
                if sim_matrix[i, j] > 0.7:
                    edge_indices.extend([[i, j], [j, i]])

        # 2. Sequential edges
        for i in range(len(titles) - 1):
            edge_indices.extend([[i, i + 1], [i + 1, i]])

        # 3. Keyword edges
        tfidf = TfidfVectorizer(stop_words = 'english', max_features = 50)
        tfidf_matrix = tfidf.fit_transform(passages.values())
        for i in range(len(titles)):
            for j in range(i + 1, len(titles)):
                if tfidf_matrix[i].dot(tfidf_matrix[j].T).sum() > 2:
                    edge_indices.extend([[i, j], [j, i]])

        # 4. title matching edges
        coref_edges = []
        for i, title in enumerate(titles):
            for j, other_title in enumerate(titles):
                if i != j and title.lower().split()[0] in other_title:
                    edge_indices.extend([[i, j], [j, i]])

        edge_index = torch.tensor(edge_indices).t().contiguous() if edge_indices else torch.empty((2,0), dtype=torch.long)
        edge_attr = torch.tensor([sim_matrix[i, j] for i, j in edge_indices]) #Edge weight for GCN (Add while GNN_GCN)

        return Data(x = node_features, edge_index = edge_index), titles

dataset = WikiHopDataset(train_data[:2000])

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.4k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# GNN Implementations

In [10]:
# This is the GNN(GRAPHSAGE) model
class RetrievalGNN_GRAPHSAGE(torch.nn.Module):
  def __init__(self, input_dim = 768, hidden_dim = 256):
    super().__init__()
    self.gnn = GraphSAGE(
        in_channels = input_dim,
        hidden_channels = hidden_dim,
        num_layers = 2,
        aggr = 'mean'  # Best from experiments
    )
    self.proj = torch.nn.Linear(hidden_dim, input_dim)  # 256 -> 768 (For matching with question embedding)
  def forward(self, data):
    x = self.gnn(data.x, data.edge_index)
    return self.proj(x)

graphsage_model = RetrievalGNN_GRAPHSAGE().to('cpu')

In [ ]:
class RetrievalGNN_GCN(torch.nn.Module):
    def __init__(self, input_dim = 768, hidden_dim = 256):
        super().__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.proj = torch.nn.Linear(hidden_dim, input_dim)

    def forward(self, data):
        x = self.conv1(data.x, data.edge_index)
        x = torch.relu(x)
        x = self.conv2(x, data.edge_index)
        return self.proj(x)

gcn_model = RetrievalGNN_GCN().to('cpu')

In [ ]:
class RetrievalGNN_GAT(torch.nn.Module):
    def __init__(self, input_dim = 768, hidden_dim = 256):
        super().__init__()
        self.conv1 = GATConv(input_dim, hidden_dim)
        self.conv2 = GATConv(hidden_dim, hidden_dim)
        self.proj = torch.nn.Linear(hidden_dim, input_dim)

    def forward(self, data):
        x = self.conv1(data.x, data.edge_index)
        x = torch.relu(x)
        x = self.conv2(x, data.edge_index)
        return self.proj

gat_model = RetrievalGNN_GAT().to('cpu')

In [ ]:
class RetrievalGNN_GAT_MultiHead(torch.nn.Module):
    def __init__(self, input_dim=768, hidden_dim=256):
        super().__init__()
        self.conv1 = GATConv(input_dim, hidden_dim, heads=3, aggr = 'mean')  # Multihead attention
        self.conv2 = GATConv(hidden_dim*3, hidden_dim, aggr = 'mean')  # We need to concatenate all the heads
        self.proj = torch.nn.Linear(hidden_dim, input_dim)
        self.dropout = torch.nn.Dropout(0.3)

    def forward(self, data):
        x = F.relu(self.conv1(data.x, data.edge_index))
        x = self.dropout(x)
        x = self.conv2(x, data.edge_index)
        return self.proj(x)
gat_model_multihead = RetrievalGNN_GAT_MultiHead().to('cpu')

In [ ]:
class RetrievalGNN_GATv2(torch.nn.Module):
    def __init__(self, input_dim=768, hidden_dim=256):
        super().__init__()
        self.conv1 = GATv2Conv(input_dim, hidden_dim)
        # self.conv1 = GATv2Conv(input_dim, hidden_dim, heads = 4, concat = True, aggr = 'mean')
        self.conv2 = GATv2Conv(hidden_dim, hidden_dim)
        self.proj = torch.nn.Linear(hidden_dim, input_dim)

    def forward(self, data):
        x = self.conv1(data.x, data.edge_index)
        x = torch.relu(x)
        x = self.conv2(x, data.edge_index)
        return self.proj(x)

gat_2Conv_model = RetrievalGNN_GATv2().to('cpu')

In [11]:
def contrastive_loss(question_embedding, passage_embeddings, positives, negatives, margin = 0.2):
    if len(positives) == 0 or len(negatives) == 0:
        return torch.tensor(0.0, device = question_embedding.device)

    pos_sim = torch.cosine_similarity(question_embedding, passage_embeddings[positives])
    neg_sim = torch.cosine_similarity(question_embedding, passage_embeddings[negatives])

    loss_matrix = torch.relu(margin - pos_sim.unsqueeze(1) + neg_sim.unsqueeze(0))

    return loss_matrix.mean()


# def contrastive_loss(question_embedding, passage_embeddings, positives, negatives, margin = 0.2):
#     if len(positives) == 0 or len(negatives) == 0:
#         return torch.tensor(0.0, device=question_embedding.device)

#     pos_sim = torch.cosine_similarity(question_embedding, passage_embeddings[positives])
#     neg_sim = torch.cosine_similarity(question_embedding, passage_embeddings[negatives])

#     hardest_negative = torch.max(neg_sim)
#     loss = torch.relu(margin - pos_sim + hardest_negative)
#     return loss.mean()

# Training

In [12]:
gnn = graphsage_model.to(device)
optimizer = torch.optim.Adam(gnn.parameters(), lr = 1e-4)
# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)
sbert = SentenceTransformer('all-mpnet-base-v2') # Taken from Assignment 1
num_epochs = 10
for epoch in range(num_epochs):
  with tqdm(dataset.data, desc = f"Epoch {epoch + 1}/{num_epochs}", unit = "sample") as tepoch:
    for sample in tepoch:
        graph, titles = dataset.build_graph(sample['context'])
        graph = graph.to(device)

        question_embedding = torch.tensor(sbert.encode(sample['question'])).unsqueeze(0).to(device)

        # Forward pass
        updated_embs = gnn(graph)


        pos_titles = [sf[0] for sf in sample['supporting_facts']]
        pos_mask = [t in pos_titles for t in titles]
        positives = torch.where(torch.tensor(pos_mask))[0]
        negatives = torch.where(~torch.tensor(pos_mask))[0]

        # Hard negative mining
        with torch.no_grad():
          sim_scores = torch.cosine_similarity(question_embedding, updated_embs)
          num_pos = len(positives)
          num_neg_samples = min(3 * num_pos, len(negatives))
          hard_negatives = negatives[torch.topk(sim_scores[negatives].cpu(), k = num_neg_samples, largest = False).indices]
          # hard_negatives = negatives[torch.argsort(sim_scores[negatives].cpu())[-3:]]  # Top 3 hardest

        # Compute loss
        loss = contrastive_loss(question_embedding, updated_embs, positives, hard_negatives)
        # print(loss)

        # Backprop
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(gnn.parameters(), 1.0)
        optimizer.step()
        # scheduler.step()

        tepoch.set_postfix(loss = loss.item())
        del graph

Epoch 1/10:   0%|          | 1/2000 [00:15<8:35:56, 15.49s/sample, loss=0.205]


KeyboardInterrupt: 

# BaseLine Models

In [ ]:
class BM25Retriever:
    def __init__(self):
        self.tokenize = lambda text: text.lower().split()

    def retrieve(self, question, context, k = 5):
        passages = [' '.join(sents) for title, sents in context]
        bm25 = BM25Okapi([self.tokenize(p) for p in passages])
        scores = bm25.get_scores(self.tokenize(question))
        return np.argsort(scores)[-k:][::-1]

class DPRRetriever:
    def __init__(self):
        self.model = SentenceTransformer('all-mpnet-base-v2')

    def retrieve(self, question, context, k = 5):
        passages = [' '.join(sents) for title, sents in context]
        q_embed = self.model.encode(question)
        p_embeds = self.model.encode(passages)
        scores = np.inner(q_embed, p_embeds)
        return np.argsort(scores)[-k:][::-1]

# Evalution Retriever

In [ ]:
# This function is used just to evaluate the retrivals
def evaluate_retriever(retriever, data_path, retriever_type = 'gnn'):
    dataset = WikiHopDataset(data_path)
    mrr = []
    f1_scores = []

    for sample in dataset.data[:100]:
        passages = [' '.join(sents) for title, sents in sample['context']]
        titles = [title for title, sents in sample['context']]
        question = sample['question']

        if retriever_type == 'gnn':
            graph, _ = dataset.build_graph(sample['context'])
            graph = graph.to(device)
            with torch.no_grad():
                updated_embs = retriever(graph)
            question_embedding = torch.tensor(dataset.sbert.encode(question)).to(device)
            scores = torch.cosine_similarity(question_embedding, updated_embs)
            ranked_indices = torch.argsort(scores, descending = True).cpu().numpy()
        elif retriever_type == 'bm25':
          ranked_indices = retriever.retrieve(question, sample['context'])
        elif retriever_type == 'dpr':
          ranked_indices = retriever.retrieve(question, sample['context'])

        ranked_titles = [titles[i] for i in ranked_indices]
        gold_titles = [sf[0] for sf in sample['supporting_facts']]

        relevant = set(gold_titles)
        retrieved = set(ranked_titles[:5])
        precision = len(relevant & retrieved) / 5
        recall = len(relevant & retrieved) / len(relevant)
        f1 = 2 * (precision * recall) / (precision + recall + 1e-8)
        f1_scores.append(f1)

        for rank, title in enumerate(ranked_titles, 1):
            if title in gold_titles:
                mrr.append(1/rank)
                break

    return np.mean(f1_scores), np.mean(mrr)

# Ouestion Answer System

In [ ]:
class QARAG:
    def __init__(self):
        self.gnn_retriever = gnn.to('cpu')
        self.bm25 = BM25Retriever()
        self.dpr = DPRRetriever()

        # Initialize T5
        # self.t5 = T5ForConditionalGeneration.from_pretrained('t5-base').to('cpu')
        # self.tokenizer = T5Tokenizer.from_pretrained('t5-base')

        self.tokenizer = AutoTokenizer.from_pretrained('google/flan-t5-base')
        self.t5 = AutoModelForSeq2SeqLM.from_pretrained('google/flan-t5-base').to('cpu')

        self.sbert = SentenceTransformer('all-mpnet-base-v2')

    def _format_prompt(self, question, contexts):
        return (
            f"Answer the question based on the context. "
            f"Question: {question}\n"
            f"Contexts:\n- " + '\n- '.join(contexts) + "\n"
            f"Answer:"
        )

        # return f"Question: {question}\nContext: {' '.join(contexts)}\nAnswer:"

    def generate_answer(self, question, context, retriever_type = 'gnn', k = 5):
        if retriever_type == 'gnn':
            self.gnn_retriever = self.gnn_retriever.to(device)
            graph, titles = dataset.build_graph(context)
            graph = graph.to(device)
            with torch.no_grad():
                updated_embs = self.gnn_retriever(graph)
            self.gnn_retriever = self.gnn_retriever.to("cpu")
            graph = graph.to('cpu')
            torch.cuda.empty_cache()
            question_emb = torch.tensor(self.sbert.encode(question)).to(device)
            scores = torch.cosine_similarity(question_emb, updated_embs)
            topk_indices = torch.topk(scores, k = k).indices.cpu().numpy()
            contexts = [' '.join(context[i][1]) for i in topk_indices]
            del graph
        elif retriever_type == 'bm25':
            indices = self.bm25.retrieve(question, context, k = k)
            contexts = [' '.join(context[i][1]) for i in indices]
        else:  # dpr
            indices = self.dpr.retrieve(question, context, k = k)
            contexts = [' '.join(context[i][1]) for i in indices]

        input_text = self._format_prompt(question, contexts)
        self.t5 = self.t5.to(device)

        # input_ids = self.tokenizer.encode(input_text, return_tensors = 'pt').to(device)
        # outputs = self.t5.generate(input_ids, max_length = 128)

        input_ids = self.tokenizer.encode(
            input_text,
            return_tensors = 'pt',
            max_length = 512,
            truncation = True
        ).to(device)

        outputs = self.t5.generate(
            input_ids,
            max_length = 200,
            num_beams = 4,
            early_stopping = True,
            repetition_penalty = 2.5,
            length_penalty = 1.2
        )
        self.t5 = self.t5.to('cpu')
        torch.cuda.empty_cache()

        answer = self.tokenizer.decode(outputs[0], skip_special_tokens = True)
        answer = answer.lower().strip()
        if not answer:
          answer = "no answer"

        return answer

In [ ]:
# This code is for LLM answer evaluation
def evaluate_answers(qa_system, data_path, retriever_type = 'gnn'):
    dataset = WikiHopDataset(data_path)
    rouge = Rouge()
    f1_scores = []
    rouge_scores = []

    for sample in dataset.data[:500]:
        generated = qa_system.generate_answer(
            sample['question'],
            sample['context'],
            retriever_type = retriever_type
        )

        torch.cuda.empty_cache()

        # Get gold answer
        gold = sample['answer']

        # print(f"Generated Answer: {generated}")
        # print(f"Gold Answer: {gold}")
        # print(f"Retriever_type: {retriever_type}")

        # Calculate F1
        gold_tokens = set(gold.lower().split())
        gen_tokens = set(generated.lower().split())
        overlap = len(gold_tokens & gen_tokens)
        precision = overlap / len(gen_tokens) if gen_tokens else 0
        recall = overlap / len(gold_tokens) if gold_tokens else 0
        f1 = 2 * (precision * recall) / (precision + recall + 1e-8)
        f1_scores.append(f1)

        # Calculate ROUGE-L
        scores = rouge.get_scores(generated, gold)
        rouge_scores.append(scores[0]['rouge-l']['f'])

    return np.mean(f1_scores), np.mean(rouge_scores)

# Retriever Evalution

In [ ]:
bm25 = BM25Retriever()
dpr = DPRRetriever()

# Evaluation Retrieval
print("Evaluating Retrieval Performance:")
for name, retriever, model_type in [('GNN', gnn, 'gnn'), ('BM25', bm25, 'bm25'), ('DPR', dpr, 'dpr')]:
    f1, mrr = evaluate_retriever(retriever, eval_data[:1000], retriever_type = model_type)
    print(f"{name}: F1@5 = {f1:.3f}, MRR = {mrr:.3f}")

# Answer Evalution

In [ ]:
qa_system = QARAG()

# Evaluation Answer Generation
print("\nEvaluating Answer Generation:")
for model_type in ['gnn', 'bm25', 'dpr']:
    f1, rouge = evaluate_answers(qa_system, eval_data[:1000], retriever_type = model_type)
    print(f"{model_type.upper()}: Answer F1 = {f1:.3f}, ROUGE-L = {rouge:.3f}")